## Imported all libraries needed to begin project and connected to sqlite database.

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path

DB_PATH = Path('output/meadowtable.db')
conn    = sqlite3.connect(DB_PATH)

LABOR_TARGET = 30.0
HOURLY_WAGE  = 14.50

print('Connected to database.')
print('Tables:', pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)['name'].tolist())

Connected to database.
Tables: ['locations', 'sales', 'labor']


## Created a summary of financial data

In [2]:
summary = pd.read_sql("""
    SELECT
        ROUND(SUM(s.revenue), 0)              AS total_revenue,
        ROUND(SUM(lb.labor_cost), 0)          AS total_labor_cost,
        ROUND(AVG(lb.labor_pct), 2)           AS avg_labor_pct,
        ROUND(SUM(lb.variance_cost), 0)       AS total_variance_cost,
        ROUND(SUM(lb.overtime_hrs), 1)        AS total_overtime_hrs,
        ROUND(SUM(s.covers), 0)               AS total_covers,
        ROUND(AVG(s.avg_check), 2)            AS avg_check
    FROM sales s
    JOIN labor lb ON s.location_id = lb.location_id
                 AND s.date        = lb.date
                 AND s.daypart     = lb.daypart
""", conn)

print('── SYSTEM-WIDE KPIs (18 months) ──────────────────────────')
for col in summary.columns:
    val = summary[col].iloc[0]
    if 'revenue' in col or 'cost' in col or 'check' in col:
        print(f'  {col:<25}: ${val:>12,.2f}')
    else:
        print(f'  {col:<25}: {val:>12}')

── SYSTEM-WIDE KPIs (18 months) ──────────────────────────
  total_revenue            : $30,101,328.00
  total_labor_cost         : $10,071,707.00
  avg_labor_pct            :        32.48
  total_variance_cost      : $   -5,604.00
  total_overtime_hrs       :     144403.5
  total_covers             :    1627732.0
  avg_check                : $       18.49


## Began running hypothosis targeting if certain days are systematically overstaffed. Used the average labor percentage against a 30% target by shift and day of the week. If a segement is consistently above 30% target it would be flagged as a scheduling pattern problem and not random noise.

In [3]:
daypart_analysis = pd.read_sql("""
    SELECT
        s.daypart,
        s.day_of_week,
        ROUND(AVG(lb.labor_pct), 2)       AS avg_labor_pct,
        ROUND(AVG(lb.variance_hrs), 2)    AS avg_variance_hrs,
        ROUND(SUM(lb.variance_cost), 0)   AS total_variance_cost,
        COUNT(*)                           AS observations
    FROM sales s
    JOIN labor lb ON s.location_id = lb.location_id
                 AND s.date        = lb.date
                 AND s.daypart     = lb.daypart
    GROUP BY s.daypart, s.day_of_week
    ORDER BY avg_labor_pct DESC
""", conn)

daypart_analysis['vs_target'] = (daypart_analysis['avg_labor_pct'] - LABOR_TARGET).round(2)
daypart_analysis['status'] = daypart_analysis['vs_target'].apply(
    lambda x: 'OVERSTAFFED' if x > 2 else ('UNDERSTAFFED' if x < -2 else 'On Target')
)

print('Labor % by Daypart and Day of Week (sorted worst first):')
print(daypart_analysis.to_string(index=False))
print(f'\nOverstaffed segments: {(daypart_analysis["status"]=="OVERSTAFFED").sum()}')
print(f'Understaffed segments: {(daypart_analysis["status"]=="UNDERSTAFFED").sum()}')

Labor % by Daypart and Day of Week (sorted worst first):
daypart day_of_week  avg_labor_pct  avg_variance_hrs  total_variance_cost  observations  vs_target      status
 Dinner    Saturday          37.06             -0.03               -595.0          1560       7.06 OVERSTAFFED
 Dinner      Sunday          36.47              0.08               1740.0          1580       6.47 OVERSTAFFED
 Dinner      Friday          35.54             -0.13              -2988.0          1560       5.54 OVERSTAFFED
 Dinner    Thursday          33.80             -0.02               -560.0          1560       3.80 OVERSTAFFED
 Dinner   Wednesday          33.28             -0.01               -183.0          1560       3.28 OVERSTAFFED
 Dinner      Monday          32.64             -0.01               -116.0          1560       2.64 OVERSTAFFED
 Dinner     Tuesday          32.60             -0.07              -1525.0          1560       2.60 OVERSTAFFED
  Lunch      Friday          30.92              0.01   

## Began a second hypothesis to see which locations have the worst labor efficiencies. Created a scorecard for each location.

In [4]:
location_scorecard = pd.read_sql("""
    SELECT
        l.location_id,
        l.store_name,
        l.market,
        l.manager_tenure_yrs,
        ROUND(SUM(s.revenue), 0)              AS total_revenue,
        ROUND(AVG(lb.labor_pct), 2)           AS avg_labor_pct,
        ROUND(SUM(lb.variance_cost), 0)       AS total_variance_cost,
        ROUND(AVG(lb.variance_hrs), 2)        AS avg_variance_hrs,
        ROUND(SUM(lb.overtime_hrs), 1)        AS total_overtime_hrs,
        ROUND(AVG(s.covers), 1)               AS avg_daily_covers
    FROM sales s
    JOIN labor lb    ON s.location_id = lb.location_id
                    AND s.date        = lb.date
                    AND s.daypart     = lb.daypart
    JOIN locations l ON s.location_id = l.location_id
    GROUP BY l.location_id, l.store_name, l.market, l.manager_tenure_yrs
    ORDER BY avg_labor_pct DESC
""", conn)

location_scorecard['vs_target'] = (location_scorecard['avg_labor_pct'] - LABOR_TARGET).round(2)
location_scorecard['flag'] = location_scorecard['vs_target'].apply(
    lambda x: 'HIGH' if x > 3 else ('LOW' if x < -3 else '')
)

print('Location Labor Scorecard (worst first):')
print(location_scorecard[['store_name','market','avg_labor_pct','vs_target',
                           'total_variance_cost','flag']].to_string(index=False))

Location Labor Scorecard (worst first):
                 store_name       market  avg_labor_pct  vs_target  total_variance_cost flag
MeadowTable Jacksonville #4 Jacksonville          38.65       8.65              61735.0 HIGH
MeadowTable Jacksonville #2 Jacksonville          37.75       7.75              63965.0 HIGH
     MeadowTable Orlando #1      Orlando          36.96       6.96              48500.0 HIGH
       MeadowTable Tampa #2        Tampa          35.42       5.42              38187.0 HIGH
       MeadowTable Tampa #3        Tampa          34.97       4.97              24675.0 HIGH
       MeadowTable Miami #1        Miami          34.90       4.90              29465.0 HIGH
MeadowTable Jacksonville #3 Jacksonville          33.75       3.75              12084.0 HIGH
       MeadowTable Miami #4        Miami          33.45       3.45              16430.0 HIGH
       MeadowTable Miami #3        Miami          33.05       3.05               5772.0 HIGH
MeadowTable Jacksonville #1 Ja

## Analyzed ROI. I wanted to see how much each location saves by fixing overstaffing problems.

In [5]:
overstaffed = location_scorecard[location_scorecard['vs_target'] > 1.0].copy()

overstaffed['recoverable_cost_18mo'] = overstaffed['total_variance_cost'].clip(lower=0)
overstaffed['recoverable_cost_annual'] = (overstaffed['recoverable_cost_18mo'] * (12/18)).round(0)

total_recoverable_annual = overstaffed['recoverable_cost_annual'].sum()
total_overstaffed_locs   = len(overstaffed)

print('── ROI ANALYSIS: Overstaffing Impact ─────────────────────────────────')
print(f'  Overstaffed locations       : {total_overstaffed_locs} of {len(location_scorecard)}')
print(f'  Total recoverable (annual)  : ${total_recoverable_annual:,.0f}')
print(f'  Avg per location (annual)   : ${total_recoverable_annual/max(total_overstaffed_locs,1):,.0f}')
print()
print('Top 5 Highest-Impact Locations to Address:')
print(overstaffed.nlargest(5, 'recoverable_cost_annual')[
    ['store_name','market','avg_labor_pct','vs_target','recoverable_cost_annual']
].to_string(index=False))

── ROI ANALYSIS: Overstaffing Impact ─────────────────────────────────
  Overstaffed locations       : 13 of 20
  Total recoverable (annual)  : $212,931
  Avg per location (annual)   : $16,379

Top 5 Highest-Impact Locations to Address:
                 store_name       market  avg_labor_pct  vs_target  recoverable_cost_annual
MeadowTable Jacksonville #2 Jacksonville          37.75       7.75                  42643.0
MeadowTable Jacksonville #4 Jacksonville          38.65       8.65                  41157.0
     MeadowTable Orlando #1      Orlando          36.96       6.96                  32333.0
       MeadowTable Tampa #2        Tampa          35.42       5.42                  25458.0
       MeadowTable Miami #1        Miami          34.90       4.90                  19643.0


## Began another hypothesis seeing if a managers tenure correlated with greater scheduling efficiency.

In [6]:
from scipy import stats

corr, pval = stats.pearsonr(
    location_scorecard['manager_tenure_yrs'],
    location_scorecard['avg_labor_pct']
)

print('── HYPOTHESIS: Manager Tenure vs Labor % ──────────────────')
print(f'  Pearson r  : {corr:.3f}')
print(f'  p-value    : {pval:.3f}')
print()
if pval < 0.05:
    direction = 'negative' if corr < 0 else 'positive'
    print(f'  FINDING: Statistically significant {direction} correlation.')
    if corr < 0:
        print('  Longer-tenured managers tend to schedule more efficiently (lower labor %).')
    else:
        print('  Longer-tenured managers tend to overschedule slightly.')
else:
    print('  No statistically significant relationship found (p > 0.05).')
    print('  Labor inefficiency is not explained by manager tenure alone.')

── HYPOTHESIS: Manager Tenure vs Labor % ──────────────────
  Pearson r  : -0.042
  p-value    : 0.862

  No statistically significant relationship found (p > 0.05).
  Labor inefficiency is not explained by manager tenure alone.


## Looked over monthly trends in revenue and labor costs over time.

In [7]:
monthly_trend = pd.read_sql("""
    SELECT
        s.year_month,
        ROUND(SUM(s.revenue), 0)             AS total_revenue,
        ROUND(SUM(lb.labor_cost), 0)         AS total_labor_cost,
        ROUND(AVG(lb.labor_pct), 2)          AS avg_labor_pct,
        ROUND(SUM(s.covers), 0)              AS total_covers,
        ROUND(SUM(lb.variance_cost), 0)      AS variance_cost
    FROM sales s
    JOIN labor lb ON s.location_id = lb.location_id
                 AND s.date        = lb.date
                 AND s.daypart     = lb.daypart
    GROUP BY s.year_month
    ORDER BY s.year_month
""", conn)

monthly_trend['rev_per_labor_dollar'] = (
    monthly_trend['total_revenue'] / monthly_trend['total_labor_cost']
).round(2)

print('Monthly Trend:')
print(monthly_trend.to_string(index=False))

Monthly Trend:
year_month  total_revenue  total_labor_cost  avg_labor_pct  total_covers  variance_cost  rev_per_labor_dollar
   2023-01      1472697.0          477923.0          31.67       79192.0          174.0                  3.08
   2023-02      1384535.0          452229.0          31.85       75064.0         -351.0                  3.06
   2023-03      1697469.0          567023.0          32.45       92108.0          -22.0                  2.99
   2023-04      1715607.0          579476.0          32.73       92604.0         -238.0                  2.96
   2023-05      1783631.0          601770.0          32.78       96643.0           -6.0                  2.96
   2023-06      1826835.0          626477.0          33.27       98511.0         1198.0                  2.92
   2023-07      1855827.0          633986.0          33.07       99935.0          444.0                  2.93
   2023-08      1747583.0          585840.0          32.58       94502.0         -608.0                  

## Comparing each markets revenue

In [8]:
market_summary = pd.read_sql("""
    SELECT
        l.market,
        COUNT(DISTINCT s.location_id)         AS locations,
        ROUND(SUM(s.revenue), 0)              AS total_revenue,
        ROUND(AVG(s.revenue), 0)              AS avg_daily_revenue,
        ROUND(AVG(lb.labor_pct), 2)           AS avg_labor_pct,
        ROUND(SUM(lb.variance_cost), 0)       AS total_variance_cost
    FROM sales s
    JOIN labor lb    ON s.location_id = lb.location_id
                    AND s.date        = lb.date
                    AND s.daypart     = lb.daypart
    JOIN locations l ON s.location_id = l.location_id
    GROUP BY l.market
    ORDER BY avg_labor_pct DESC
""", conn)

market_summary['vs_target'] = (market_summary['avg_labor_pct'] - LABOR_TARGET).round(2)
print('Market Comparison:')
print(market_summary.to_string(index=False))

Market Comparison:
      market  locations  total_revenue  avg_daily_revenue  avg_labor_pct  total_variance_cost  vs_target
Jacksonville          4      5815591.0             1329.0          35.73             156369.0       5.73
       Tampa          4      6330486.0             1447.0          33.23              20085.0       3.23
       Miami          4      5799041.0             1325.0          33.06              39861.0       3.06
     Orlando          5      7934264.0             1451.0          31.60             -73189.0       1.60
 Tallahassee          3      4221946.0             1286.0          27.81            -148731.0      -2.19


## Saved results to be shown in a dashboard.

In [9]:

monthly_trend.to_csv('output/monthly_trend.csv', index=False)
location_scorecard.to_csv('output/location_scorecard.csv', index=False)
market_summary.to_csv('output/market_summary.csv', index=False)
daypart_analysis.to_csv('output/daypart_analysis.csv', index=False)
overstaffed.to_csv('output/overstaffed.csv', index=False)

kpis = {
    'total_revenue':          summary['total_revenue'].iloc[0],
    'avg_labor_pct':          summary['avg_labor_pct'].iloc[0],
    'total_variance_cost':    summary['total_variance_cost'].iloc[0],
    'total_overtime_hrs':     summary['total_overtime_hrs'].iloc[0],
    'recoverable_annual':     total_recoverable_annual,
    'overstaffed_locations':  total_overstaffed_locs,
}
pd.Series(kpis).to_csv('output/kpis.csv', header=False)

conn.close()
print('All outputs saved.')

All outputs saved. Ready for 03_dashboard.py
